# Fine-tuning CrossEncoder Reranker on TechQA

**BEFORE RUNNING:** Settings → Accelerator → GPU T4

**Dataset Structure:**
- `full_technote_collection.sections.json`: `{id: {id, text}}`
- `training_Q_A.json` + `dev_Q_A.json`: Q&A pairs

In [ ]:
!pip install -q sentence-transformers torch

In [ ]:
import json
import random
import os
import torch
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from torch.utils.data import DataLoader

print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU NOT FOUND')

random.seed(42)

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────
BASE = '/kaggle/input/datasets/yiannisman/techqa-dataset/TechQA'

TECHNOTES_PATH = f'{BASE}/technote_corpus/full_technote_collection.sections.json'
TRAIN_QA_PATH  = f'{BASE}/training_and_dev/training_Q_A.json'
DEV_QA_PATH    = f'{BASE}/training_and_dev/dev_Q_A.json'

for path in [TECHNOTES_PATH, TRAIN_QA_PATH, DEV_QA_PATH]:
    exists = os.path.exists(path)
    status = 'CHECK' if exists else 'FAIL'
    size   = f'{os.path.getsize(path)/1024/1024:.1f} MB' if exists else 'NOT FOUND'
    print(f'{status} {os.path.basename(path)}: {size}')

In [ ]:
# -- Step 1: Loading Technotes --------------------------------
# Structure: {"swg123": {"id": "swg123", "text": "..."}, ...}

print('Loading Technotes...')
technotes = {}

with open(TECHNOTES_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

for doc_id, doc_data in raw.items():
    technotes[doc_id] = doc_data.get('text', '')

print(f'{len(technotes)} Technotes loaded.')
sample_id = list(technotes.keys())[0]

In [ ]:
# -- Step 2: Loading Q&A pairs -------------------------------

def load_qa(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('Loading Q&A pairs...')
train_data = load_qa(TRAIN_QA_PATH)
dev_data   = load_qa(DEV_QA_PATH)

all_qa = [
    item for item in train_data + dev_data
    if item.get('ANSWERABLE') == 'Y'
    and item.get('DOCUMENT')
    and item.get('DOCUMENT') in technotes
]

missing = [
    item for item in train_data + dev_data
    if item.get('ANSWERABLE') == 'Y'
    and item.get('DOCUMENT')
    and item.get('DOCUMENT') not in technotes
]

print(f'Train questions:           {len(train_data)}')
print(f'Dev questions:             {len(dev_data)}')
print(f'Answerable με valid doc:   {len(all_qa)}')
print(f'Missing doc: {len(missing)}')

In [ ]:
# -- Step 3: Creating Training Pairs -----------------------------

NUM_NEGATIVES    = 4
all_technote_ids = list(technotes.keys())

train_samples = []
skipped       = 0

for item in all_qa:
    question   = str(item.get('QUESTION_TEXT') or item.get('QUESTION_TITLE') or '').strip()
    correct_id = item['DOCUMENT']

    if not question:
        skipped += 1
        continue

    # Positive pair
    train_samples.append(InputExample(
        texts=[question, technotes[correct_id]],
        label=1.0
    ))

    # Negative pairs
    neg_ids = random.sample(
        [tid for tid in all_technote_ids if tid != correct_id],
        NUM_NEGATIVES
    )
    for neg_id in neg_ids:
        train_samples.append(InputExample(
            texts=[question, technotes[neg_id]],
            label=0.0
        ))

random.shuffle(train_samples)

valid_q = len(all_qa) - skipped
print(f'Total training samples: {len(train_samples)}')
print(f'  Positive pairs: {valid_q}')
print(f'  Negative pairs: {valid_q * NUM_NEGATIVES}')
print(f'  Skipped:        {skipped}')

In [ ]:
# -- Step 4: Train/Validation split (90/10) -----------------------

split_idx   = int(len(train_samples) * 0.9)
train_split = train_samples[:split_idx]
val_split   = train_samples[split_idx:]

print(f'Training samples:   {len(train_split)}')
print(f'Validation samples: {len(val_split)}')

In [ ]:
# -- Step 5: Fine-tuning ─-----------------------------------------------------

MODEL_NAME  = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
OUTPUT_PATH = '/kaggle/working/techqa-reranker'
EPOCHS      = 3
BATCH_SIZE  = 16

print(f'Loading base model: {MODEL_NAME}...')
model = CrossEncoder(MODEL_NAME, num_labels=1, max_length=512)

train_dataloader = DataLoader(train_split, shuffle=True, batch_size=BATCH_SIZE)
warmup_steps     = int(len(train_dataloader) * EPOCHS * 0.1)

print(f'Training steps/epoch: {len(train_dataloader)}')
print(f'Warmup steps:         {warmup_steps}')
print(f'Starting fine-tuning for {EPOCHS} epochs...')

# IMPORTANT: save_best_model=False — manual saving afterwards
model.fit(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    save_best_model=False
)

print(f'\n Fine-tuning Complete!')

In [ ]:
# -- Step 6: Saving Model ----------------------------------------

print(f'Saving model at {OUTPUT_PATH}...')
os.makedirs(OUTPUT_PATH, exist_ok=True)
model.save(OUTPUT_PATH)

print(f'\n Files saved:')
for fname in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(os.path.join(OUTPUT_PATH, fname)) / (1024*1024)
    print(f'  {fname}: {size:.1f} MB')

In [ ]:
# -- ΒΗΜΑ 7: Evaluation (MRR@10) ---------------------------------------

print('Evaluation fine-tuned model...')

val_queries = {}
for sample in val_split:
    q, doc, lbl = sample.texts[0], sample.texts[1], sample.label
    if q not in val_queries:
        val_queries[q] = {'query': q, 'positive': [], 'negative': []}
    if lbl == 1.0:
        val_queries[q]['positive'].append(doc)
    else:
        val_queries[q]['negative'].append(doc)

evaluator_samples = [
    v for v in val_queries.values()
    if len(v['positive']) > 0 and len(v['negative']) > 0
]

print(f'Evaluation queries: {len(evaluator_samples)}')
evaluator = CrossEncoderRerankingEvaluator(evaluator_samples, name='val')
scores    = evaluator(model)

print(f'\n Evaluation Results:')
for key, value in scores.items():
    print(f'   {key}: {value:.4f}')

In [ ]:
# -- ΒΗΜΑ 8: Deployment Manual ------------------------------------------

print('=' * 60)
print('DONE! Next Steps:')
print()
print('1. Output panel (right) -> techqa-reranker -> Download')
print('2. Unzip Locally')
print('3. Copy to Docker:')
print('   docker cp ./techqa-reranker knowledge_api:/app/reranker_model')
print('4. Restart:')
print('   docker restart knowledge_api')
print('5. Verify with logs:')
print('   docker logs knowledge_api | grep reranker')
print('   -> YOU MUST SEE: Fine-tuned reranker ready!')
print('=' * 60)